In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch import Tensor
from tqdm import tqdm
import math
from sklearn.model_selection import train_test_split
from src.transformer import GPT
from src.datasets import NamesDataset, PoetryDataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, processors, decoders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
cur_dir: str = "./"

### Names dataset

In [5]:
f = open('./data/names.txt', 'r')
names = f.read().splitlines()

In [6]:
train_names, val_names = train_test_split(names, test_size=0.1, random_state=42)

train_dataset = NamesDataset(train_names, seq_len=10)
val_dataset = NamesDataset(val_names, seq_len=10)

In [ ]:
for i in range(5):
	print(train_dataset.get_sample(i))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [7]:
test_name = 'pre-ln_ffn_swiglu_sin-pe'
epochs = 50
dict_size = 29
emb_size = 8
seq_len = 10
num_heads = 8
hidden_size = 64
head_size = 64
dropout = 0.3

In [8]:
model = GPT(
	dict_size=dict_size,
	emb_size=emb_size, seq_len=seq_len,
	hidden_size=hidden_size, num_heads=num_heads,
	head_size=head_size,
	dropout=dropout,
).to(device)

In [9]:
total_params = sum(p.numel() for p in model.parameters())
total_params

105693

In [ ]:
optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=train_dataset.stoi["<pad>"])

In [ ]:
train_loss_history = []
train_perplexity_history = []
val_loss_history = []
val_perplexity_history = []

In [ ]:
for epoch in range(epochs):
	model.train()
	total_loss = 0
	for input, target in tqdm(train_loader):
		input, target = input.to(device), target.to(device)
		optim.zero_grad()
		logits = model(input)
		loss = criterion(
			logits.view(-1, logits.size(-1)),
			target.view(-1)
		)
		loss.backward()
		optim.step()
		total_loss += loss.item()

	avg_loss = total_loss / len(train_loader)
	perplexity = math.exp(avg_loss)
	train_loss_history.append(avg_loss)
	train_perplexity_history.append(perplexity)

	with torch.no_grad():
		model.eval()
		total_val_loss = 0
		for val_input, val_target in tqdm(val_loader):
			val_input, val_target = val_input.to(device), val_target.to(device)
			val_logits = model(val_input)
			val_loss = criterion(
				val_logits.view(-1, val_logits.size(-1)),
				val_target.view(-1)
			)
			total_val_loss += val_loss.item()

		avg_val_loss = total_val_loss / len(val_loader)
		val_perplexity = math.exp(avg_val_loss)
		val_loss_history.append(avg_val_loss)
		val_perplexity_history.append(val_perplexity)

	print(f"Epoch {epoch + 1} / {epochs}, Loss: {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}")

In [ ]:
train_perplexity = torch.tensor(train_perplexity_history)
torch.save(train_perplexity, cur_dir + f'outputs/{test_name}_train_perplexity_history.pt')

val_perplexity = torch.tensor(val_perplexity_history)
torch.save(val_perplexity, cur_dir + f'outputs/{test_name}_val_perplexity_history.pt')

train_loss = torch.tensor(train_loss_history)
torch.save(train_loss, cur_dir + f'outputs/{test_name}_train_loss_history.pt')

val_loss = torch.tensor(val_loss_history)
torch.save(val_loss, cur_dir + f'outputs/{test_name}_val_loss_history.pt')

In [ ]:
torch.save(model.state_dict(), cur_dir + f'weights/gpt-name-{test_name}')

In [ ]:
state_dict = torch.load('./weights/gpt-name-pre-ln_relu_sin-pe', weights_only=True, map_location=device)
model.load_state_dict(state_dict)

In [ ]:
def generate_name(model):
	model.eval()
	itos = train_dataset.itos
	start_token = train_dataset.stoi['<start>']
	end_token = train_dataset.stoi['<end>']
	idx = torch.tensor([[start_token]], device=device)
	inference = model.generate(idx, num_tokens=train_dataset.seq_len)
	tokens = inference[0].tolist()

	name = []
	for token in tokens[1:]:
		if token == end_token:
			break
		name.append(itos[token])
	return ''.join(name)

In [ ]:
for _ in range(10):
	print(generate_name(model))

### Poetry dataset

In [14]:
df = pd.read_csv(cur_dir + 'data/russianPoetryWithTheme.csv')

In [15]:
df = df[['author', 'text']]

In [16]:
df.head()

,author,text
0,Михаил Лермонтов,"Забывши волнения жизни мятежной,\r\nОдин жил в..."
1,Сергей Есенин,"Нивы сжаты, рощи голы,\r\nОт воды туман и сыро..."
2,Игорь Северянин,Лючинь печальная читала вечером ручьисто-вкрад...
3,Анатолий Жигулин,"Глыбу кварца разбили молотом,\r\nИ, веселым ог..."
4,Николай Тихонов,"Хлынул дождь, когда девушки, встав в хоровод,\..."


In [17]:
alphabet = ''.join(chr(i) for i in range(ord('а'), ord('я') + 1)) + ''.join(chr(i) for i in range(ord('A'), ord('Я') + 1))
alphabet += ' \n.,!?-«»()—…:;\"\''

In [18]:
df['text'] = df['text'].apply(lambda x: ''.join([ch for ch in x if ch in alphabet]))

In [19]:
df.head()

,author,text
0,Михаил Лермонтов,"Забывши волнения жизни мятежной,\nОдин жил в п..."
1,Сергей Есенин,"Нивы сжаты, рощи голы,\nОт воды туман и сырост..."
2,Игорь Северянин,Лючинь печальная читала вечером ручьисто-вкрад...
3,Анатолий Жигулин,"Глыбу кварца разбили молотом,\nИ, веселым огне..."
4,Николай Тихонов,"Хлынул дождь, когда девушки, встав в хоровод,\..."


In [20]:
def train_bpe_tokenizer(texts, vocab_size=8000):
    tokenizer = Tokenizer(models.BPE())
    
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    
    tokenizer.decoder = decoders.ByteLevel()
    
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<start>", "<end>"],
        min_frequency=2,
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )

    tokenizer.train_from_iterator(texts, trainer=trainer)
    
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
    
    tokenizer.enable_padding(
        pad_id=tokenizer.token_to_id("<pad>"),
        pad_token="<pad>"
    )
    return tokenizer

In [21]:
tokenizer = train_bpe_tokenizer(
    df['text'].tolist(),
    vocab_size=8000,
)

In [22]:
encoded = tokenizer.encode(df['text'][0])

In [23]:
decoded = tokenizer.decode(encoded.ids)
decoded

'Забывши волнения жизни мятежной,\nОдин жил в пустыне рыбак молодой.\nОднажды на скале прибрежной,\nНад тихой прозрачной рекой\nОн с удой беспечно\nСидел\nИ думой сердечной\nК прошедшему счастью летел.'

In [24]:
df['encoded'] = df['text'].apply(lambda x: tokenizer.encode(x).ids)

In [27]:
test_name = 'poetry_post-ln_relu_sin-pe'
epochs = 50
emb_size = 256
num_heads = 8
max_len = 512
hidden_size = 384

In [28]:
train, val = train_test_split(df, test_size=0.05, random_state=42)
train_dataset = PoetryDataset(train, tokenizer, max_len=max_len)
val_dataset = PoetryDataset(val, tokenizer, max_len=max_len)

In [29]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [30]:
model = GPT(
	dict_size=tokenizer.get_vocab_size(),
	emb_size=emb_size, seq_len=max_len,
	hidden_size=hidden_size, num_heads=num_heads,
	head_size=emb_size // num_heads,
	dropout=0.5).to(device)

In [31]:
total_params = sum(p.numel() for p in model.parameters())
total_params

6999104

In [200]:
optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id('<pad>'))

In [201]:
train_loss_history = []
train_perplexity_history = []
val_loss_history = []
val_perplexity_history = []

In [ ]:
for epoch in range(epochs):
	model.train()
	total_loss = 0
	for input, target in tqdm(train_loader):
		input, target = input.to(device), target.to(device)
		optim.zero_grad()
		logits = model(input)
		loss = criterion(
			logits.view(-1, logits.size(-1)),
			target.view(-1)
		)
		loss.backward()
		optim.step()
		total_loss += loss.item()

	avg_loss = total_loss / len(train_loader)
	perplexity = math.exp(avg_loss)
	train_loss_history.append(avg_loss)
	train_perplexity_history.append(perplexity)

	with torch.no_grad():
		model.eval()
		total_val_loss = 0
		for val_input, val_target in tqdm(val_loader):
			val_input, val_target = val_input.to(device), val_target.to(device)
			val_logits = model(val_input)
			val_loss = criterion(
				val_logits.view(-1, val_logits.size(-1)),
				val_target.view(-1)
			)
			total_val_loss += val_loss.item()

		avg_val_loss = total_val_loss / len(val_loader)
		val_perplexity = math.exp(avg_val_loss)
		val_loss_history.append(avg_val_loss)
		val_perplexity_history.append(val_perplexity)

	print(f"Epoch {epoch + 1} / {epochs}, Loss: {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}")

100%|██████████| 7/7 [00:01<00:00,  5.31it/s]


Epoch 3 / 50, Loss: 5.8721, Perplexity: 354.9760


100%|██████████| 7/7 [00:01<00:00,  5.32it/s]


Epoch 4 / 50, Loss: 5.8126, Perplexity: 334.4834


100%|██████████| 7/7 [00:01<00:00,  5.31it/s]


...

100%|██████████| 7/7 [00:01<00:00,  5.30it/s]


Epoch 48 / 50, Loss: 4.7350, Perplexity: 113.8587


100%|██████████| 7/7 [00:01<00:00,  5.34it/s]


Epoch 49 / 50, Loss: 4.7312, Perplexity: 113.4346


 92%|█████████▏| 114/124 [00:57<00:05,  1.99it/s]

In [209]:
train_perplexity = torch.tensor(train_perplexity_history)
torch.save(train_perplexity, cur_dir + f'outputs/{test_name}_train_perplexity_history.pt')

val_perplexity = torch.tensor(val_perplexity_history)
torch.save(val_perplexity, cur_dir + f'outputs/{test_name}_val_perplexity_history.pt')

train_loss = torch.tensor(train_loss_history)
torch.save(train_loss, cur_dir + f'outputs/{test_name}_train_loss_history.pt')

val_loss = torch.tensor(val_loss_history)
torch.save(val_loss, cur_dir + f'outputs/{test_name}_val_loss_history.pt')

In [210]:
torch.save(model.state_dict(), cur_dir + f'weights/gpt-{test_name}')

In [85]:
state_dict = torch.load('./weights/gpt-poetry_post-ln_relu_sin-pe', weights_only=True, map_location=device)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [86]:
def generate_poetry(model, tokenizer, max_tokens=1000):
    model.eval()
    
    bos_token_id = tokenizer.token_to_id("<start>")
    eos_token_id = tokenizer.token_to_id("<end>")
    pad_token_id = tokenizer.token_to_id("<pad>")
    
    idx = torch.tensor([[bos_token_id]], device=device)
    
    with torch.no_grad():
        for _ in tqdm(range(max_tokens)):
            idx_crop = idx if idx.shape[1] <= model.max_seq_len else idx[:, -model.max_seq_len:]
            
            logits = model(idx_crop)[:, -1, :]
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            token_id = next_token.item()
            
            if token_id == eos_token_id:
                break
            
            if token_id == pad_token_id:
                continue
            
            idx = torch.cat([idx, next_token], dim=1)
    
    generated_ids = idx[0].tolist()[1:]
    poetry = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return poetry

In [132]:
poetry = generate_poetry(model, tokenizer, max_tokens=64)

100%|██████████| 64/64 [00:00<00:00, 289.51it/s]


In [133]:
print(poetry)


И в звон зубах лед и гроз,
И мухомзы и озноб
Зорь, чуть ослепленный зазвон.
Лежит в печчитаньи шваль.
Возванит, то летит роман,
Весь высохший саблею блеснет,
И в зелени
